<a href="https://colab.research.google.com/github/IdoAbram/Tiny-NMT/blob/dev/teacher_nmt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

creatin teacher model for en-es translate

In [1]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch: 2.9.0+cu126
cuda available: True
gpu: Tesla T4


Loading Helsinki-NLP

In [2]:
import torch
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-en-es"

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
teacher = MarianMTModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher = teacher.to(device)
teacher.eval()

print("Loaded teacher on:", device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loaded teacher on: cuda


Lets start with a simple translate check

In [3]:
def translate_teacher(texts, max_length=128, num_beams=4):
    if isinstance(texts, str):
        texts = [texts]
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = teacher.generate(**inputs, max_length=max_length, num_beams=num_beams)
    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

tests = [
    "Hello! How are you today?",
    "I want to build a very small translation model.",
    "This is a simple test sentence."
]

for src, tgt in zip(tests, translate_teacher(tests)):
    print("EN:", src)
    print("ES:", tgt)
    print("-" * 40)


EN: Hello! How are you today?
ES: Hola, ¿cómo estás hoy?
----------------------------------------
EN: I want to build a very small translation model.
ES: Quiero construir un modelo de traducción muy pequeño.
----------------------------------------
EN: This is a simple test sentence.
ES: Esta es una simple frase de prueba.
----------------------------------------


Getting the size of the model

In [4]:
def model_size_info(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    # גודל משוער ב-MB לפי dtype הנוכחי של הפרמטרים
    bytes_per_param = next(model.parameters()).element_size()
    size_mb = total_params * bytes_per_param / (1024**2)
    return total_params, trainable_params, size_mb, bytes_per_param

total, trainable, size_mb, bpp = model_size_info(teacher)
print(f"Total params: {total:,}")
print(f"Trainable params: {trainable:,}")
print(f"Estimated size: {size_mb:.1f} MB (bytes/param={bpp})")


Total params: 77,943,296
Trainable params: 77,943,296
Estimated size: 297.3 MB (bytes/param=4)


simple benchmark

In [5]:
import time

sample_texts = ["This is a speed test sentence."] * 64  # batch קטן
# warmup
_ = translate_teacher(sample_texts, num_beams=4)

start = time.time()
_ = translate_teacher(sample_texts, num_beams=4)
elapsed = time.time() - start

# ספירה גסה של tokens בכניסה (לא מושלם, אבל טוב להתחלה)
tok = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True)
n_tokens = tok["input_ids"].numel()
print(f"Elapsed: {elapsed:.3f}s | approx input tokens: {n_tokens} | approx tokens/sec: {n_tokens/elapsed:.0f}")


Elapsed: 0.267s | approx input tokens: 512 | approx tokens/sec: 1915


Lets load data so we could calculate benchmark and BELU

In [13]:
from datasets import load_dataset, get_dataset_config_names

DATASET_ID = "Helsinki-NLP/opus_books"
print(get_dataset_config_names(DATASET_ID)[:20])  # תראה שיש "en-es"

ds = load_dataset(DATASET_ID, "en-es")
print(ds)
print("splits:", list(ds.keys()))
print("example:", ds["train"][0])

['ca-de', 'ca-en', 'ca-hu', 'ca-nl', 'de-en', 'de-eo', 'de-es', 'de-fr', 'de-hu', 'de-it', 'de-nl', 'de-pt', 'de-ru', 'el-en', 'el-es', 'el-fr', 'el-hu', 'en-eo', 'en-es', 'en-fi']
DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 93470
    })
})
splits: ['train']
example: {'id': '0', 'translation': {'en': 'Source: Project GutenbergAudiobook available here', 'es': 'Source: Wikisource & librodot.com'}}


Preparing the text to eval

In [15]:
def get_src_tgt(split, src_lang="en", tgt_lang="es", limit=None):
    src, tgt = [], []
    for ex in split:
        tr = ex["translation"]
        src.append(tr[src_lang])
        tgt.append(tr[tgt_lang])
        if limit and len(src) >= limit:
            break
    return src, tgt

train_src, train_ref = get_src_tgt(ds["train"], "en", "es", limit=None)
print("train size:", len(train_src))
print("sample:", train_src[0], "=>", train_ref[0])


train size: 93470
sample: Source: Project GutenbergAudiobook available here => Source: Wikisource & librodot.com


Benchmarking :) N = 500

In [18]:
import time

N = 500

def batched_translate(texts, batch_size=32, num_beams=4, max_length=128):
    outputs = []
    total_time = 0.0
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        t0 = time.time()
        out = translate_teacher(batch, num_beams=num_beams, max_length=max_length)
        t1 = time.time()
        outputs.extend(out)
        total_time += (t1 - t0)
    return outputs, total_time

eval_src = train_src[:N]

preds, total_seconds = batched_translate(
    eval_src,
    batch_size=32,
    num_beams=4
)

print("sentences:", len(preds))
print("total seconds:", round(total_seconds, 2))
print("sentences/sec:", round(len(preds) / total_seconds, 2))


sentences: 500
total seconds: 36.04
sentences/sec: 13.87


BELU + chrF + TER

In [20]:
!pip install sacrebleu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.6 MB/s eta 0:00:00


In [21]:
from sacrebleu.metrics import BLEU, CHRF, TER

N = 500

eval_ref = train_ref[:N]

bleu = BLEU()
chrf = CHRF()
ter  = TER()

bleu_score = bleu.corpus_score(preds, [eval_ref])
chrf_score = chrf.corpus_score(preds, [eval_ref])
ter_score  = ter.corpus_score(preds, [eval_ref])

print("BLEU:", bleu_score)
print("chrF:", chrf_score)
print("TER :", ter_score)

BLEU: BLEU = 27.59 59.3/33.3/21.2/13.9 (BP = 1.000 ratio = 1.004 hyp_len = 14545 ref_len = 14488)
chrF: chrF2 = 53.49
TER : TER = 60.19
